## 1. Change Dataset Format


In [7]:
import pandas as pd
import os

KATEGORI_ROWS = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']


def clean_harga(value):
    """Konversi string harga ke float. '-' atau kosong jadi NaN."""
    if pd.isna(value) or str(value).strip() in ['-', '']:
        return None
    try:
        return float(str(value).replace(',', '').strip())
    except:
        return None


def process_sheet(df, wilayah):
    df = df[~df['No'].isin(KATEGORI_ROWS)].copy()
    df = df.reset_index(drop=True)

    kolom_tanggal = [c for c in df.columns if c not in ['No', 'Komoditas (Rp)']]

    df_long = df.melt(
        id_vars=['No', 'Komoditas (Rp)'],
        value_vars=kolom_tanggal,
        var_name='tanggal_raw',
        value_name='harga'
    )

    df_long['wilayah']   = wilayah
    df_long['komoditas'] = df_long['Komoditas (Rp)'].str.strip()
    df_long['harga']     = df_long['harga'].apply(clean_harga)
    df_long['tanggal']   = pd.to_datetime(
        df_long['tanggal_raw'].str.strip(),
        format='%d/ %m/ %Y',
        errors='coerce'
    )

    df_long = df_long[['tanggal', 'wilayah', 'komoditas', 'harga']]

    # Hanya hapus kalau tanggalnya kosong, harga biarkan NaN
    df_long = df_long.dropna(subset=['tanggal'])
    df_long = df_long.sort_values(['komoditas', 'tanggal'])
    df_long = df_long.reset_index(drop=True)

    return df_long


def run_etl(
    input_path="../data/raw/TabelHargaGabungan.xlsx",
    output_path="../data/processed/harga_gabungan.csv"
):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    all_sheets = pd.read_excel(input_path, sheet_name=None)
    all_dfs = []

    for sheet_name, df in all_sheets.items():
        wilayah = sheet_name.replace("Tabel Harga", "").strip()
        df_long = process_sheet(df, wilayah)
        all_dfs.append(df_long)

    df_final = pd.concat(all_dfs, ignore_index=True)
    df_final = df_final.sort_values(['tanggal', 'wilayah', 'komoditas'])
    df_final = df_final.reset_index(drop=True)

    df_final.to_csv(output_path, index=False)

    print(f"ETL selesai → {len(df_final)} baris disimpan ke {output_path}")

    return df_final


if __name__ == "__main__":
    df = run_etl()
    print("\nPreview hasil:")
    print(df.head(10).to_string(index=False))

ETL selesai → 18392 baris disimpan ke ../data/processed/harga_gabungan.csv

Preview hasil:
   tanggal wilayah                  komoditas  harga
2019-01-01    Aceh Bawang Merah Ukuran Sedang    NaN
2019-01-01    Aceh Bawang Putih Ukuran Sedang    NaN
2019-01-01    Aceh     Beras Kualitas Bawah I    NaN
2019-01-01    Aceh    Beras Kualitas Bawah II    NaN
2019-01-01    Aceh    Beras Kualitas Medium I    NaN
2019-01-01    Aceh   Beras Kualitas Medium II    NaN
2019-01-01    Aceh     Beras Kualitas Super I    NaN
2019-01-01    Aceh    Beras Kualitas Super II    NaN
2019-01-01    Aceh          Cabai Merah Besar    NaN
2019-01-01    Aceh       Cabai Merah Keriting    NaN
